# Daily Challenge: End-to-End Web Scraping in Python
Scraping the **GitHub Topics** page (<https://github.com/topics>) with `requests`, `BeautifulSoup` and `pandas`.

**Pipeline:** fetch HTML → check status → save to `webpage.html` → parse → extract titles & descriptions → dictionary → DataFrame.

## 0. Setup

In [ ]:
!pip install requests beautifulsoup4 pandas --quiet

In [ ]:
import re

import requests
import pandas as pd
from bs4 import BeautifulSoup

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)

## 1. Fetch the HTML content with `requests`
We send a browser-like `User-Agent` header, since some sites reject the default one used by `requests`.

In [ ]:
URL = "https://github.com/topics"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/124.0 Safari/537.36",
    "Accept-Language": "en-US,en;q=0.9",
}

response = requests.get(URL, headers=HEADERS, timeout=30)

# Print the status code (it should be 200 if the request was successful)
print("Status code:", response.status_code)

# Stop early with a clear error if the request failed (4xx / 5xx)
response.raise_for_status()

In [ ]:
# Print the first 100 characters of the HTML to verify the response
print(response.text[:100])

## 2. Save the HTML to `webpage.html`
We write the file with explicit **UTF-8** encoding so special characters are stored correctly (no `UnicodeEncodeError`, no garbled text).

In [ ]:
# Make sure `requests` decodes the body as UTF-8 (GitHub serves UTF-8 pages)
response.encoding = "utf-8"

# Save the HTML content to a file, specifying the encoding explicitly
with open("webpage.html", "w", encoding="utf-8") as file:
    file.write(response.text)

print("Saved webpage.html")

## 3. Parse the saved HTML with BeautifulSoup

In [ ]:
# Read the saved file back (same encoding used when writing it)
with open("webpage.html", "r", encoding="utf-8") as file:
    html_content = file.read()

soup = BeautifulSoup(html_content, "html.parser")

# Quick sanity check: the page <title>
print("Page title:", soup.title.get_text(strip=True))

## 4. Identify the pieces of information to extract
We will extract **two distinct pieces of information** for every featured topic:
1. **Title** (e.g. "Python")
2. **Description** (e.g. "Python is a dynamically typed programming language.")

Each topic is rendered as a card that is a link (`<a href="/topics/<name>">`) containing the title and the description in separate elements. First, let's inspect one card to identify its tags and classes.

In [ ]:
# Find the topic-card links: <a href="/topics/<topic-name>">
TOPIC_HREF = re.compile(r"^(https://github\.com)?/topics/[\w\-.]+/?$")

# Keep only links that contain text (the image-only links have none)
card_link = next(a for a in soup.find_all("a", href=TOPIC_HREF) if len(list(a.stripped_strings)) >= 2)

# Inspect the HTML tags and classes used inside one card
print(card_link.prettify()[:1500])

The title and the description are stored in separate child elements (tags/classes) inside the link. Rather than depending on CSS class names (which GitHub changes often), we take the text pieces of each card with `.stripped_strings`: the **last** piece is the description and the **one before it** is the title (some cards include an extra short label before the title; this approach ignores it).

## 5. Extract the titles and descriptions

In [ ]:
topic_titles = []
topic_descriptions = []
seen_links = set()          # avoid duplicates: some topics are shown twice on the page

for link in soup.find_all("a", href=TOPIC_HREF):
    slug = link["href"].rstrip("/").split("/")[-1]    # e.g. "python" (works for absolute and relative URLs)
    texts = list(link.stripped_strings)               # all text fragments inside the card link

    # Skip image-only links, the plain "Topics" links, and duplicated cards
    if len(texts) < 2 or slug in seen_links:
        continue
    seen_links.add(slug)

    topic_titles.append(texts[-2])            # title = second to last text fragment
    topic_descriptions.append(texts[-1])      # description = last text fragment

# Verify the extraction: length and content of each list
print("Number of titles:", len(topic_titles))
print("Titles:", topic_titles)
print()
print("Number of descriptions:", len(topic_descriptions))
for desc in topic_descriptions:
    print("-", desc)

# Both lists must have the same length so they can form a table
assert len(topic_titles) == len(topic_descriptions), "Lists have different lengths!"

## 6. Build a dictionary and convert it to a pandas DataFrame

In [ ]:
# Dictionary: keys = type of information, values = the extracted lists
topics_dict = {
    "title": topic_titles,
    "description": topic_descriptions,
}

# Convert the dictionary into a pandas DataFrame
topics_df = pd.DataFrame(topics_dict)

# Print the DataFrame to confirm its structure and contents
print(topics_df)
print("\nShape:", topics_df.shape)
topics_df.info()

In [ ]:
# (Optional) display it as a nice table and save it to CSV for later analysis
display(topics_df)
topics_df.to_csv("github_topics.csv", index=False, encoding="utf-8")
print("Saved github_topics.csv")

## 7. Quick analysis (bonus)

In [ ]:
# Longest descriptions and average description length
topics_df["description_length"] = topics_df["description"].str.len()
print("Average description length:", round(topics_df["description_length"].mean(), 1), "characters")
display(topics_df.sort_values("description_length", ascending=False).head(5))

## Notes
- **Why `status_code` matters:** 200 means success; 403/429 usually means the site is blocking or rate-limiting automated requests.
- **Why save the HTML first:** we can re-parse the file as many times as we like without hitting the website again.
- **Fragile selectors:** if GitHub changes its page layout, only the extraction cell (section 5) needs to be adjusted.
- Always respect the site's `robots.txt`, terms of service and rate limits.